# task_11 Solution

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

base = Path("../data")


In [ ]:

meters = pd.read_csv(base / "meters.csv")
usage = pd.read_csv(base / "usage.csv")
tariffs = pd.read_csv(base / "tariffs.csv")
outages = pd.read_csv(base / "outages.csv")

usage["timestamp"] = pd.to_datetime(usage["timestamp"])
outages["outage_start"] = pd.to_datetime(outages["outage_start"])

usage["dow"] = usage["timestamp"].dt.dayofweek
usage["hour"] = usage["timestamp"].dt.hour

peak = usage[(usage["dow"] < 5) & (usage["hour"] >= 18) & (usage["hour"] <= 21)]
q1 = str(peak.groupby("meter_id")["kwh"].mean().sort_values(ascending=False).index[0])

rate_map = {}
for row in tariffs.itertuples(index=False):
    for hour in range(int(row.start_hour), int(row.end_hour) + 1):
        rate_map[hour] = row.rate_per_kwh
usage["rate"] = usage["hour"].map(rate_map)
weekend = usage[usage["dow"] >= 5]
q2 = f"{(weekend['kwh'] * weekend['rate']).sum():.2f}"

daily_total = usage.groupby(usage["timestamp"].dt.date)["kwh"].sum().sort_index()
outage_dates = sorted({d.date() for d in outages["outage_start"]})
drops = {}
day_index = list(daily_total.index)
for day in outage_dates:
    if day in daily_total.index:
        idx = day_index.index(day)
        if idx > 0:
            prev = daily_total.iloc[idx - 1]
            curr = daily_total.loc[day]
            if prev != 0:
                drops[day] = (prev - curr) / prev * 100
q3 = max(drops.items(), key=lambda x: x[1])[0].isoformat()

meter_daily = usage.groupby(["meter_id", usage["timestamp"].dt.date])["kwh"].sum().reset_index(name="daily_kwh")
meter_daily.rename(columns={"timestamp": "date"}, inplace=True)
outage_set = set(outage_dates)
meter_daily["is_outage"] = meter_daily["date"].isin(outage_set)
reductions = []
for meter_id, group in meter_daily.groupby("meter_id"):
    outage_avg = group[group["is_outage"]]["daily_kwh"].mean()
    normal_avg = group[~group["is_outage"]]["daily_kwh"].mean()
    reductions.append((meter_id, normal_avg - outage_avg))
q4 = sorted(reductions, key=lambda x: (-x[1], x[0]))[0][0]

by_type = usage.merge(meters, on="meter_id").groupby(["household_type", "timestamp"])["kwh"].sum().reset_index()
load_factor = by_type.groupby("household_type")["kwh"].agg(["mean", "max"])
load_factor["factor"] = load_factor["mean"] / load_factor["max"]
q5 = str(load_factor["factor"].sort_values(ascending=False).index[0])


Q1: Which meter_id has the highest average kwh during weekday evening peak hours 18:00 through 21:59?

In [ ]:
print(q1)


Q2: What is the total billed amount on weekend days, using the tariff rate determined by each usage row's hour, rounded to 2 decimals?

In [ ]:
print(q2)


Q3: Among the outage dates, which date had the largest percentage drop in total aggregate kwh versus the previous date?

In [ ]:
print(q3)


Q4: Which meter_id had the largest reduction in average daily kwh on outage dates compared with that meter's own average daily kwh on non-outage dates?

In [ ]:
print(q4)


Q5: Which household_type has the higher load factor, defined as mean aggregate hourly kwh divided by max aggregate hourly kwh over the full period?

In [ ]:
print(q5)
